## Read Me

This is a flexible notebook for stepping through an annotation step by step given a specific scenario input. The notebook must be modified in order to select the directory and input file. This is not ideal, so we will have to come up with a better system!

## Set up

In [1]:
import json # !pip
import os
import textwrap
import pandas as pd
import importlib
import numpy as np
from pathlib import Path
import sys

In [2]:
ROOT_DIR = os.getcwd() + '/../'
sys.path.append(ROOT_DIR)

sys.path.append(ROOT_DIR+'src/')
print(ROOT_DIR)

/Users/anna/Dropbox/2025_moral_scenario_annotation/code/anna/graph_extract/run_annotation/../


In [3]:
import src.annotate_scenario as annotate_scenario
import src.prompts as prompts
import src.translate_to_vis as translate_to_vis
import src.node as node
import src.get_emb_distances as get_emb_distances
import src.utils as utils
import src.moral_projection as moral_projection

import src.core_process as core_process
importlib.reload(annotate_scenario)
importlib.reload(translate_to_vis)
importlib.reload(prompts)
importlib.reload(node)
importlib.reload(utils)
importlib.reload(core_process)

<module 'src.core_process' from '/Users/anna/Dropbox/2025_moral_scenario_annotation/code/anna/graph_extract/run_annotation/../src/core_process.py'>

In [4]:
# Automatically reload modules when they change
%load_ext autoreload
%autoreload 2

## Selet Scenario Input File

In [5]:
# set main paths
SCENARIO_DIR = ROOT_DIR + "scenarios_inputs/" + "cheung_variants/"
# DATA_DIR_HUMAN = ROOT_DIR + "human_data/" 
OUTPUT_DIR = ROOT_DIR + "annotated_outputs/" + "cheung_variants/"

In [6]:
#set scenario file filename
FILENAME = 'bird.json'

#select scenario and action choice
SCENARIO_ID = 4
ACT_ID = '1'

#read in the scenario
scenario_json = utils.open_scenario(SCENARIO_DIR, FILENAME, SCENARIO_ID, ACT_ID)


Scenario Text: 


When I was 9 or 10, I had a BB gun I would shoot in our backyard on weekends. One morning, I was
shooting at a target. When I was about to shoot, a bird started flying by. I didn’t notice the bird
and pulled the trigger as it entered my vision. What happened next seemed like slow motion. The bird
fell from flight and I started hearing it frantically wriggling in pain on the ground. I froze for a
minute as it slowly started to sink in that I had shot the bird in one of its wings. I frantically
ran over. I felt traumatized because I loved animals, especially birds, and I didn’t want to hurt
them. I started to cry because I didn’t know what to do next. I knew the bird was in pain and that
it would likely lose the shot wing. I saw it living in pain and suffering for a long time unless I
acted. I had a shovel that I could use to end the bird's life. 




In [7]:
# print scenario json
print(json.dumps(scenario_json, indent=4))

{
    "id": 4,
    "scenario_title": "Bird",
    "deontology_level": "1",
    "utility_level": "2",
    "text": "When I was 9 or 10, I had a BB gun I would shoot in our backyard on weekends. One morning, I was shooting at a target. When I was about to shoot, a bird started flying by. I didn\u2019t notice the bird and pulled the trigger as it entered my vision. What happened next seemed like slow motion. The bird fell from flight and I started hearing it frantically wriggling in pain on the ground. I froze for a minute as it slowly started to sink in that I had shot the bird in one of its wings. I frantically ran over. I felt traumatized because I loved animals, especially birds, and I didn\u2019t want to hurt them. I started to cry because I didn\u2019t know what to do next. I knew the bird was in pain and that it would likely lose the shot wing. I saw it living in pain and suffering for a long time unless I acted. I had a shovel that I could use to end the bird's life.",
    "options"

#### There are 4 major stages of processing.

0. Entities
Label the entities (no human data)

1. Value Scores / "Deontology"
Score the action in moral value

2. Outcomes
Map action to probable outcomes 

3. Outcome Utilities
Consequentialist analysis of harms/benefits of each outcome to each entity

4. Outcome Links
Connection between each entity and each outcome, in terms of Cause, Intend, and Desire



## Go through annotation process step by step (replicates main function in annotate_scenario) 

In [8]:

# get the action choice and convert to two pronoun options (I and Ziv)
this_act = scenario_json['options'][ACT_ID]
this_act_I = this_act
this_act_Ziv = annotate_scenario.prompts.convert_I_Ziv(this_act_I)
print('\n\nAction choice:') 
print(this_act_Ziv)
print(this_act_I)

#get the scenario and convert to two pronoun options
this_scenario = scenario_json['text']
print(scenario_json['text'])
this_scenario_Ziv = annotate_scenario.prompts.convert_I_Ziv(this_scenario)
print("\n\nScenario:")
print(this_scenario_Ziv)
print(this_scenario)


# create a dictionary to write out to csv later
scenario_dict = {'scenario': this_scenario, 'scenario_idx': scenario_json['id'],
                    'choice': this_act_I}





Action choice:
Kill the bird with the shovel.
kill the bird with the shovel
When I was 9 or 10, I had a BB gun I would shoot in our backyard on weekends. One morning, I was shooting at a target. When I was about to shoot, a bird started flying by. I didn’t notice the bird and pulled the trigger as it entered my vision. What happened next seemed like slow motion. The bird fell from flight and I started hearing it frantically wriggling in pain on the ground. I froze for a minute as it slowly started to sink in that I had shot the bird in one of its wings. I frantically ran over. I felt traumatized because I loved animals, especially birds, and I didn’t want to hurt them. I started to cry because I didn’t know what to do next. I knew the bird was in pain and that it would likely lose the shot wing. I saw it living in pain and suffering for a long time unless I acted. I had a shovel that I could use to end the bird's life.


Scenario:
When Ziv was 9 or 10, they had a BB gun they would sh

In [19]:

#initialize Graph object    
g = annotate_scenario.node.Graph()
g.reset()   
print('Graph g initialized and reset.')
g.set_version('d6d780129095bb9540a7957befb2014ea42d92c1')



Graph g initialized and reset.


#### Step 0. Get entities

In [20]:
#Step 0. Get entities

# identify all sentient beings, returning both pronoun forms and a string list
returned_beings = annotate_scenario.core_process.process_beings(this_scenario,this_act,g)
beings_I = returned_beings[0]
beings_Ziv = returned_beings[1]
beings_str_list = returned_beings[2]
g = returned_beings[3]
    
#update the scenario dict with the beings
scenario_dict["entities"] = beings_str_list



Identified these entities: 

I
1 bird


#### Step 1. Deontology / Action Value Scores

In [21]:
#Step 1.  #ACTION VALUE SCORES

#call the process_values function to rate the moral goodness or wrongness of the action with no context
deontic_value,g  = annotate_scenario.core_process.process_values_simple(this_scenario, this_act_I, this_act_I,g) 
print(deontic_value)

    

-71.0


#### Step 2. Anticipated Outcomes

In [22]:
#Step 2. Outcomes
processed_events = annotate_scenario.core_process.process_outcomes(this_scenario, this_act)
events_I= processed_events[1]
events_Ziv= processed_events[0]
print("\n".join(events_I))         
scenario_dict["outcomes"]= events_I

The bird dies
The bird's pain ends
I kill the bird with the shovel
I experience emotional distress
I avoid leaving the bird alive and suffering


#### Step 3. Outcome Utilities

In [23]:
g.print_graph()

[{'node': {'kind': 'being', 'label': 'i'},
  'links': [{'link': {'kind': 'b-link', 'value': 'C+I+K+'},
    'to_node': 'kill the bird with the shovel'}]},
 {'node': {'kind': 'being', 'label': '1 bird'}, 'links': []},
 {'node': {'kind': 'action_choice', 'label': 'kill the bird with the shovel'},
  'links': [{'link': {'kind': 'v-link', 'value': '-71.0'},
    'to_node': 'value'}]},
 {'node': {'kind': 'value', 'label': 'value'}, 'links': []},
 {'version': 'd6d780129095bb9540a7957befb2014ea42d92c1'}]

In [29]:
#Step 3. Outcome utilities

impacts_list = core_process.process_impacts(this_scenario_Ziv, this_act, this_act_Ziv, events_Ziv, events_I,beings_Ziv,g) 



Processing impacts of event: The bird dies
{'Ziv': -85, '1 bird': -100}

Processing impacts of event: The bird's pain ends
{'Ziv': 0, '1 bird': 95}

Processing impacts of event: Ziv kills the bird with the shovel
{'Ziv': -65, '1 bird': -100}

Processing impacts of event: Ziv experiences emotional distress
{'Ziv': -85, '1 bird': 0}

Processing impacts of event: Ziv avoids leaving the bird alive and suffering
{'Ziv': 72, '1 bird': 85}


#### Step 4. Cause / Intend / Know Links

In [41]:
#Step 4. causal / intentional / knowledge links -- run on currently generated event/outcome list
output_links = annotate_scenario.process_causal_links(this_scenario_Ziv, events_Ziv, events_I, this_act_Ziv,g)    


Processing event: The man on the ladder is shot and dies.
{'cause': 'yes', 'intend': 'yes', 'know': 'yes'}
CKI links for I
C+I+K+

Processing event: The man falls off the ladder.
{'cause': 'yes', 'intend': 'yes', 'know': 'yes'}
CKI links for I
C+I+K+

Processing event: The 19 other passengers behind the man are able to climb onto the deck.
{'cause': 'yes', 'intend': 'yes', 'know': 'no'}
CKI links for I
C+I+K-

Processing event: I experience the psychological impact of shooting the man.
{'cause': 'yes', 'intend': 'no', 'know': 'no'}
CKI links for I
C+I-K-

Processing event: The other passengers witness the shooting.
{'cause': 'yes', 'intend': 'no', 'know': 'yes'}
CKI links for I
C+I-K+

Processing event: The overall chance of survival for the group on the ferry increases.
{'cause': 'yes', 'intend': 'yes', 'know': 'no'}
CKI links for I
C+I+K-


#### Step 5. Write out the results

In [42]:
narrative = FILENAME.split('.')[0]
this_output_filename = f"{OUTPUT_DIR}/{narrative}_{SCENARIO_ID}_choice_{ACT_ID}.json"

In [43]:
#optional -- write out the results 
print('\n\nWriting to file: '+this_output_filename)
g_print = g.print_graph()
utils.write_jsonlines(this_output_filename, g_print)
print('\n\n')


translate_to_vis.main(this_output_filename)




Writing to file: /Users/anna/Dropbox/2025_moral_scenario_annotation/code/anna/graph_extract/run_annotation/../annotated_outputs/cheung_variants//rope_ladder_1_choice_1.json



/Users/anna/Dropbox/2025_moral_scenario_annotation/code/anna/graph_extract/run_annotation/../annotated_outputs/cheung_variants//rope_ladder_1_choice_1.json
